# 04 · Two-Stage Reasoning

The paper's third strategy (Sec. 3.4): two independently trained models.

1. **Stage 1** — `(Q, I) → R`. Trained at the dataset's normal lr/epochs.
2. **Stage 2** — `(Q, I, R) → A`. Trained at **lr 5e-5 for 20 epochs**
   (paper Sec. 5.1).

Stage 2 consumes the rationale **generated by stage 1**, on both train and
test. Feeding gold rationales at training time and generated ones at test time
would create a train/test mismatch — and at test time gold rationales leak
the answer outright.

This notebook trains two models, so budget roughly double a single run.

In [1]:
%%writefile medthink_bn.py
"""
medthink_bn.py
==============
Shared library for reproducing MedThink (Gai et al., Findings of NAACL 2025)
on Bengali Med-VQA data: R-RAD-Bengali, R-SLAKE-Bangla, R-PathVQA-Bengali.

Faithful to the paper except for one deliberate, necessary substitution:

    UnifiedQA / T5-base  ->  csebuetnlp/banglat5   (or google/mt5-base)

The paper's TextualEncoder/Decoder is UnifiedQA (T5-base), whose SentencePiece
vocabulary contains no Bengali codepoints; Bengali input degenerates to <unk>.
BanglaT5 (247M) is Bengali-native and the closest match to the paper's 223M
budget, so its "0.2B parameters" claim still holds. Set model_name to
"google/mt5-base" for the multilingual 580M alternative.

Both are T5 v1.1 (gated-gelu, untied embeddings) and diverge to NaN at the
paper's lr of 5e-4. Trained here at 1e-4 with the visual gate initialised
closed; see MedThinkMultimodalGeneration._init_visual.

Everything else follows the paper:
  * VisualEncoder      : DETR (Carion et al. 2020) -> (100, 256) features
  * Cross-attention    : Eq. 2, softmax(QK^T/sqrt(d))V with Q from text, K/V from image
  * Gated fusion       : Eq. 3-4, lambda = sigmoid(Wl*F_T + Wv*H_attn)
  * Loss               : Eq. 6, token-level NLL
  * Strategies         : without_R, Explanation, Reasoning, Two-Stage Reasoning

Metrics are Bengali-aware and implemented from scratch (no English stemmers,
no whitespace-only tokenisation) so scores are meaningful on Bengali script.
"""

from __future__ import annotations

import json
import math
import os
import random
import re
import unicodedata
from collections import Counter
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

__all__ = [
    "DATASETS", "STRATEGIES", "Config",
    "bn_normalize", "bn_tokenize",
    "corpus_bleu", "rouge_scores", "exact_match", "token_f1", "chrf",
    "evaluate_open_end", "evaluate_open_end_dual", "evaluate_closed_end", "snap_to_choice",
    "load_split", "unify_records", "build_image_index", "resolve_image",
    "build_input_text", "build_target_text", "parse_answer",
    "MedThinkDataset", "collate_factory",
    "MedThinkMultimodalGeneration", "BanglaT5MultimodalGeneration", "load_model",
    "set_seed", "count_parameters", "diagnose_forward",
    "measure_gate", "majority_baseline", "constant_answer_baseline",
    "degenerate_rate", "record_uid", "seed_checkpoint",
    "train_model", "generate_predictions", "save_results",
]

# --------------------------------------------------------------------------- #
# 0. Dataset registry
# --------------------------------------------------------------------------- #

# Filenames differ per dataset (the Bengali release is not internally uniform).
DATASETS: Dict[str, Dict[str, Any]] = {
    "rad": {
        "dir": "R-RAD-Bengali",
        "image_field": "image",
        "splits": {
            "closed-end": {"train": "trainset_bengali.json", "test": "testset_bengali.json"},
            "open-end": {"train": "trainset_bengali.json", "test": "testset_bengali.json"},
        },
        # open-end answers are int indices into a 1-element `choices` list
        "open_answer_mode": "index",
        "epochs": 150,          # paper Sec. 5.1
    },
    "slake": {
        "dir": "R-SLAKE-Bangla",
        "image_field": "img_name",
        "splits": {
            "closed-end": {"train": "train_bengali.json", "test": "test_bengali.json"},
            "open-end": {"train": "train_bengali.json", "test": "test_bengali.json"},
        },
        "open_answer_mode": "index",
        "epochs": 300,          # paper Sec. 5.1
    },
    "path": {
        "dir": "R-PathVQA-Bengali",
        "image_field": "image",
        "splits": {
            "closed-end": {
                "train": "trainset_bengali.json",
                "val": "valset_bengali.json",
                "test": "testset_bengali.json",
            },
            "open-end": {
                "train": "trainset_bengali.json",
                "val": "valset_bengali.json",
                "test": "testset_bengali.json",
            },
        },
        # open-end answers are raw strings; `choices` is a vestigial ["হ্যাঁ","না"]
        # left over from the closed-end template and MUST be ignored.
        "open_answer_mode": "string",
        "epochs": 50,           # paper Sec. 5.1
    },
}

STRATEGIES = [
    "without_R",                # baseline: answer only
    "Explanation",              # A then R
    "Reasoning",                # R then A
    "First-Stage_Reasoning",    # (Q,I) -> R
    "Second-Stage_Reasoning",   # (Q,I,R) -> A
]

# Bengali surface markers. Using Bengali rather than the paper's English
# "The answer is (X)" keeps the target distribution inside the backbone's
# Bengali pretraining domain; the structural role is identical.
MARK_Q = "প্রশ্ন:"
MARK_OPT = "বিকল্প:"
MARK_ANS = "উত্তর:"
MARK_RAT = "কারণ:"
OPTION_LETTERS = ["ক", "খ", "গ", "ঘ", "ঙ", "চ"]


@dataclass
class Config:
    """Every knob for one training run. Serialised alongside results."""
    dataset: str = "rad"                       # rad | slake | path
    subset: str = "closed-end"                 # closed-end | open-end
    strategy: str = "Explanation"              # see STRATEGIES

    # BanglaT5 (247M) -- Bengali-native and closest to the paper's 223M, so the
    # paper's "0.2B parameters" comparison survives. Alternative:
    # "google/mt5-base" (580M, multilingual; then use batch_size=2,
    # grad_accum=16, since it needs ~9GB before activations).
    # Both are T5 v1.1 (gated-gelu, untied embeddings), an unstable family, so
    # the low lr and closed-gate init below are load-bearing, not cosmetic.
    model_name: str = "csebuetnlp/banglat5"
    patch_dim: int = 256                       # DETR hidden size
    patch_num: int = 100                       # DETR object queries

    # With dynamic padding these are truncation caps only, not the padded
    # width, so generous values cost nothing. Real inputs are 11-78 tokens and
    # targets reach 318, so nothing is actually truncated.
    source_len: int = 512
    target_len: int = 320                      # raised from 64: Bengali rationales
    #                                            average ~75 words (~150+ tokens)

    # The paper uses 5e-4, tuned for UnifiedQA (T5 v1.0). BanglaT5 is T5 v1.1
    # (feed_forward_proj="gated-gelu", tie_word_embeddings=False), which is
    # markedly less stable: 5e-4 drives it to NaN within a few steps. 1e-4 is
    # the standard fine-tuning rate for the v1.1 family. Documented deviation.
    lr: float = 1e-4
    lr_stage2: float = 5e-5                    # paper: second stage
    epochs: Optional[int] = None               # None -> paper value for dataset
    epochs_stage2: int = 20                    # paper Sec. 5.1
    # 8 x 4 = the paper's effective batch of 32. Dynamic padding cut real
    # sequence widths from 512 to ~30, so a larger micro-batch now fits and
    # keeps the GPU better occupied. For mT5-base (580M) use 4 x 8.
    batch_size: int = 8
    grad_accum: int = 4                        # effective batch 32 (paper)
    weight_decay: float = 1e-2
    warmup_ratio: float = 0.06
    label_smoothing: float = 0.0
    max_grad_norm: float = 1.0

    # DataLoader workers. Kaggle is Linux and forks cheaply; Windows spawns,
    # which needs an `if __name__ == "__main__"` guard the notebooks cannot
    # provide, so default to 0 there.
    num_workers: int = 0 if os.name == "nt" else 2

    num_beams: int = 1
    seed: int = 42
    fp16: bool = False                         # T5 overflows in fp16; keep off
    quick_run: bool = False                    # tiny subset smoke test

    data_root: str = "/kaggle/input"
    feature_dir: str = "/kaggle/input/medthink-bn-features"
    output_dir: str = "/kaggle/working"

    def resolved_epochs(self) -> int:
        if self.epochs is not None:
            return self.epochs
        return DATASETS[self.dataset]["epochs"]

    def model_tag(self) -> str:
        """
        Short backbone id, e.g. 'mt5-base' or 'banglat5'.

        Must be filename-safe: it goes into every results filename. A local
        checkpoint directory (offline Kaggle runs point `model_name` at one)
        would otherwise embed path separators and produce an unopenable path.
        """
        tag = self.model_name.replace("\\", "/").rstrip("/").split("/")[-1]
        return re.sub(r"[^A-Za-z0-9._-]+", "-", tag) or "model"

    def run_name(self) -> str:
        # The backbone is part of the identity: without it, an mT5 run and a
        # BanglaT5 run of the same dataset/strategy would write to the same
        # metrics file and silently overwrite each other.
        name = f"{self.dataset}_{self.subset}_{self.strategy}_{self.model_tag()}"
        # A quick_run is a diagnostic, not a result. Keeping it in a separate
        # namespace means re-running the sanity check after a real run cannot
        # overwrite hours of work with a 5-minute smoke test.
        return name + "_SANITY" if self.quick_run else name

    def to_json(self) -> str:
        return json.dumps(asdict(self), ensure_ascii=False, indent=2)


# --------------------------------------------------------------------------- #
# 1. Bengali normalisation + tokenisation
# --------------------------------------------------------------------------- #

# csebuetnlp's normalizer is the reference Bengali normaliser (and what
# BanglaT5 expects). Optional here: if the package
# is unavailable (offline Kaggle session) we fall back to an NFKC-based
# normaliser that handles the cases that actually matter for this corpus.
try:  # pragma: no cover - environment dependent
    from normalizer import normalize as _csebuetnlp_normalize  # type: ignore
    _HAS_CSEBUETNLP = True
except Exception:  # pragma: no cover
    _csebuetnlp_normalize = None
    _HAS_CSEBUETNLP = False

# Bengali digits -> ASCII, so "২" and "2" compare equal.
_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

# Characters that are visually identical but differently encoded.
_BN_FIXES = {
    "ড়": "ড়",   # RRA  -> DDA + nukta
    "ঢ়": "ঢ়",   # RHA  -> DHA + nukta
    "য়": "য়",   # YYA  -> YA  + nukta
    "‌": "",               # ZWNJ
    "‍": "",               # ZWJ
    "৤": "।", "৥": "।",
}

_PUNCT_RE = re.compile(r"[।,;:!?\"'`()\[\]{}<>/\\|@#$%^&*_+=~—–\-\.]+")
_WS_RE = re.compile(r"\s+")
# Bengali block + ASCII alphanumerics: medical text is code-mixed (DWI, MRI...).
_TOKEN_RE = re.compile(r"[ঀ-৿]+|[A-Za-z]+|[0-9]+")


def bn_normalize(text: str, use_csebuetnlp: bool = True) -> str:
    """Normalise Bengali text for both model input and metric computation."""
    if text is None:
        return ""
    text = str(text)
    if use_csebuetnlp and _HAS_CSEBUETNLP:
        try:
            text = _csebuetnlp_normalize(text)
        except Exception:
            pass
    text = unicodedata.normalize("NFKC", text)
    for bad, good in _BN_FIXES.items():
        text = text.replace(bad, good)
    text = text.translate(_BN_DIGITS)
    return _WS_RE.sub(" ", text).strip()


def bn_tokenize(text: str, lower: bool = True) -> List[str]:
    """
    Tokenise Bengali for BLEU/ROUGE.

    Whitespace splitting leaves punctuation glued to words ("ফুসফুস।" vs
    "ফুসফুস"), which silently deflates n-gram overlap. English tokenisers are
    worse: they stem and lowercase against a Latin alphabet. This splits on the
    Bengali Unicode block plus Latin/digit runs, which is what the scores in
    the paper's Table 3 are actually meant to measure.
    """
    text = bn_normalize(text)
    if lower:
        text = text.lower()
    return _TOKEN_RE.findall(text)


def strip_punct(text: str) -> str:
    return _WS_RE.sub(" ", _PUNCT_RE.sub(" ", text)).strip()


# --------------------------------------------------------------------------- #
# 2. Metrics (pure Python, Bengali-safe)
# --------------------------------------------------------------------------- #

def _ngrams(tokens: Sequence[str], n: int) -> Counter:
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def corpus_bleu(
    hypotheses: Sequence[str],
    references: Sequence[str],
    max_n: int = 4,
) -> Dict[str, float]:
    """
    Cumulative corpus BLEU-1..BLEU-n with brevity penalty.

    Uses the standard clipped-precision formulation. A tiny epsilon replaces a
    zero precision so that BLEU-4 does not collapse to exactly 0 on short
    single-phrase answers (common in open-end Med-VQA), matching the smoothing
    that BLEU implementations conventionally apply.
    """
    assert len(hypotheses) == len(references)
    clipped = [0] * (max_n + 1)
    total = [0] * (max_n + 1)
    hyp_len = ref_len = 0

    for hyp, ref in zip(hypotheses, references):
        h = bn_tokenize(hyp)
        r = bn_tokenize(ref)
        hyp_len += len(h)
        ref_len += len(r)
        for n in range(1, max_n + 1):
            h_ng = _ngrams(h, n)
            r_ng = _ngrams(r, n)
            total[n] += max(sum(h_ng.values()), 0)
            clipped[n] += sum(min(c, r_ng[g]) for g, c in h_ng.items())

    bp = 1.0 if hyp_len > ref_len else (math.exp(1 - ref_len / hyp_len) if hyp_len > 0 else 0.0)

    out: Dict[str, float] = {}
    log_sum = 0.0
    for n in range(1, max_n + 1):
        if total[n] == 0:
            # No n-grams of this order exist anywhere in the corpus (very short
            # answers). The order is undefined rather than wrong, so it must not
            # drag the cumulative score down: treat it as neutral.
            p = 1.0
        elif clipped[n] == 0:
            p = 1e-9        # genuine total miss -> smoothed, not division by zero
        else:
            p = clipped[n] / total[n]
        log_sum += math.log(p)
        out[f"BLEU-{n}"] = 100.0 * bp * math.exp(log_sum / n)
    return out


def _lcs_len(a: Sequence[str], b: Sequence[str]) -> int:
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(cur[j], prev[j + 1]))
        prev = cur
    return prev[-1]


def rouge_scores(hypotheses: Sequence[str], references: Sequence[str]) -> Dict[str, float]:
    """ROUGE-1/2/L F1, averaged over examples, on Bengali tokens."""
    r1 = r2 = rl = 0.0
    n = max(len(hypotheses), 1)

    for hyp, ref in zip(hypotheses, references):
        h, r = bn_tokenize(hyp), bn_tokenize(ref)

        for k, acc in ((1, "r1"), (2, "r2")):
            h_ng, r_ng = _ngrams(h, k), _ngrams(r, k)
            overlap = sum(min(c, r_ng[g]) for g, c in h_ng.items())
            hs, rs = sum(h_ng.values()), sum(r_ng.values())
            p = overlap / hs if hs else 0.0
            rc = overlap / rs if rs else 0.0
            f = 2 * p * rc / (p + rc) if (p + rc) else 0.0
            if acc == "r1":
                r1 += f
            else:
                r2 += f

        l = _lcs_len(h, r)
        p = l / len(h) if h else 0.0
        rc = l / len(r) if r else 0.0
        rl += 2 * p * rc / (p + rc) if (p + rc) else 0.0

    return {"ROUGE-1": 100.0 * r1 / n, "ROUGE-2": 100.0 * r2 / n, "ROUGE-L": 100.0 * rl / n}


def exact_match(hypotheses: Sequence[str], references: Sequence[str]) -> float:
    hits = sum(
        1
        for h, r in zip(hypotheses, references)
        if strip_punct(bn_normalize(h)).lower() == strip_punct(bn_normalize(r)).lower()
    )
    return 100.0 * hits / max(len(hypotheses), 1)


def token_f1(hypotheses: Sequence[str], references: Sequence[str]) -> float:
    tot = 0.0
    for h, r in zip(hypotheses, references):
        ht, rt = Counter(bn_tokenize(h)), Counter(bn_tokenize(r))
        overlap = sum(min(c, rt[g]) for g, c in ht.items())
        hs, rs = sum(ht.values()), sum(rt.values())
        p = overlap / hs if hs else 0.0
        rc = overlap / rs if rs else 0.0
        tot += 2 * p * rc / (p + rc) if (p + rc) else 0.0
    return 100.0 * tot / max(len(hypotheses), 1)


def chrf(hypotheses: Sequence[str], references: Sequence[str], n: int = 6, beta: float = 2.0) -> float:
    """chrF++-style character n-gram F-score. Suits agglutinative Bengali."""
    tot = 0.0
    for hyp, ref in zip(hypotheses, references):
        h = bn_normalize(hyp).replace(" ", "")
        r = bn_normalize(ref).replace(" ", "")
        fs = []
        for k in range(1, n + 1):
            hg, rg = _ngrams(h, k), _ngrams(r, k)
            overlap = sum(min(c, rg[g]) for g, c in hg.items())
            hs, rs = sum(hg.values()), sum(rg.values())
            if hs == 0 or rs == 0:
                continue
            p, rc = overlap / hs, overlap / rs
            if p + rc == 0:
                fs.append(0.0)
            else:
                b2 = beta ** 2
                fs.append((1 + b2) * p * rc / (b2 * p + rc))
        tot += sum(fs) / len(fs) if fs else 0.0
    return 100.0 * tot / max(len(hypotheses), 1)


def evaluate_open_end(hypotheses: Sequence[str], references: Sequence[str]) -> Dict[str, float]:
    """Full open-end metric suite (paper Table 3 + correctness diagnostics)."""
    out: Dict[str, float] = {}
    out.update(rouge_scores(hypotheses, references))
    out.update(corpus_bleu(hypotheses, references))
    out["ExactMatch"] = exact_match(hypotheses, references)
    out["TokenF1"] = token_f1(hypotheses, references)
    out["chrF"] = chrf(hypotheses, references)
    return out


def evaluate_open_end_dual(
    full_preds: Sequence[str],
    full_refs: Sequence[str],
    answer_preds: Sequence[str],
    answer_refs: Sequence[str],
) -> Dict[str, float]:
    """
    Open-end scoring at two granularities.

    `full_*`   : whole generation vs whole target (answer + rationale).
    `answer_*` : parsed answer span vs gold answer only.

    Both are reported because they measure different things and the paper is
    ambiguous about which it used. Table 3 must be the *full* variant: R-SLAKE
    open-end answers average 1-2 tokens, which admits no 4-grams at all, so the
    reported BLEU-4 of 10.3 cannot come from answer-only scoring. The
    answer-only numbers are the clinically meaningful ones -- a model can score
    well on rationale overlap while getting the diagnosis wrong.
    """
    out: Dict[str, float] = {}
    for k, v in evaluate_open_end(full_preds, full_refs).items():
        out[f"full/{k}"] = v
    for k, v in evaluate_open_end(answer_preds, answer_refs).items():
        out[f"ans/{k}"] = v
    return out


def evaluate_closed_end(pred_texts: Sequence[str], gold_texts: Sequence[str],
                        choices: Sequence[Sequence[str]]) -> Dict[str, float]:
    """
    Accuracy for closed-end questions (paper Table 2).

    Generation is free-form, so the prediction is snapped to the nearest option
    before scoring: exact match first, then substring, then token-overlap.
    """
    correct = 0
    unresolved = 0
    for pred, gold, opts in zip(pred_texts, gold_texts, choices):
        choice = snap_to_choice(pred, opts)
        if choice is None:
            unresolved += 1
        elif strip_punct(bn_normalize(choice)).lower() == strip_punct(bn_normalize(gold)).lower():
            correct += 1
    n = max(len(pred_texts), 1)
    return {
        "Accuracy": 100.0 * correct / n,
        "Unresolved": 100.0 * unresolved / n,
    }


def snap_to_choice(pred: str, options: Sequence[str]) -> Optional[str]:
    """Map a free-form generation onto one of the answer options."""
    p = strip_punct(bn_normalize(pred)).lower()
    norm_opts = [strip_punct(bn_normalize(o)).lower() for o in options]

    for o, no in zip(options, norm_opts):
        if p == no:
            return o
    # Prefer the option appearing earliest in the generation.
    best, best_pos = None, len(p) + 1
    for o, no in zip(options, norm_opts):
        if no and no in p:
            pos = p.index(no)
            if pos < best_pos:
                best, best_pos = o, pos
    if best is not None:
        return best

    pt = Counter(bn_tokenize(p))
    best, best_score = None, 0.0
    for o, no in zip(options, norm_opts):
        ot = Counter(bn_tokenize(no))
        overlap = sum(min(c, ot[g]) for g, c in pt.items())
        denom = sum(ot.values())
        score = overlap / denom if denom else 0.0
        if score > best_score:
            best, best_score = o, score
    return best if best_score > 0 else None


# --------------------------------------------------------------------------- #
# 3. Data loading and schema unification
# --------------------------------------------------------------------------- #

def load_split(data_root: str, dataset: str, subset: str, split: str) -> Dict[str, dict]:
    """Load one raw Bengali JSON file. Returns {id: record}."""
    meta = DATASETS[dataset]
    fname = meta["splits"][subset].get(split)
    if fname is None:
        raise KeyError(f"{dataset}/{subset} has no '{split}' split")

    root = Path(data_root)
    candidates = list(root.glob(f"**/{meta['dir']}/{subset}/{fname}"))
    if not candidates:
        candidates = list(root.glob(f"**/{subset}/{fname}"))
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {meta['dir']}/{subset}/{fname} under {data_root}. "
            f"Attach the Bengali dataset to the notebook."
        )
    with open(candidates[0], encoding="utf-8") as fh:
        return json.load(fh)


def unify_records(raw: Dict[str, dict], dataset: str, subset: str, split: str) -> List[dict]:
    """
    Flatten the three inconsistent schemas into one record type.

    The three releases disagree in ways that silently corrupt results if
    ignored:
      * R-RAD / R-SLAKE open-end : `answer` is an int index into `choices`
      * R-PathVQA open-end       : `answer` is the answer STRING, and `choices`
                                   is a leftover ["হ্যাঁ","না"] that is wrong
      * R-SLAKE                  : image key is `img_name`, not `image`
    """
    meta = DATASETS[dataset]
    img_field = meta["image_field"]
    open_mode = meta["open_answer_mode"]
    out: List[dict] = []
    skipped = 0

    for qid, rec in raw.items():
        question = bn_normalize(rec.get("question", ""))
        choices = [bn_normalize(c) for c in rec.get("choices", []) or []]
        ans = rec.get("answer")

        if subset == "closed-end":
            if not isinstance(ans, int) or not choices or ans >= len(choices):
                skipped += 1
                continue
            answer_text = choices[ans]
            options = choices
        else:
            if open_mode == "string":
                if not isinstance(ans, str) or not ans.strip():
                    skipped += 1
                    continue
                answer_text = bn_normalize(ans)
                options = []          # vestigial choices deliberately dropped
            else:
                if not isinstance(ans, int) or not choices or ans >= len(choices):
                    skipped += 1
                    continue
                answer_text = choices[ans]
                options = []

        solution = bn_normalize(rec.get("solution", "") or "")
        image_key = rec.get(img_field) or rec.get("image") or rec.get("img_name") or ""

        out.append({
            "qid": str(qid),
            "dataset": dataset,
            "subset": subset,
            "split": split,
            "question": question,
            "answer": answer_text,
            "options": options,
            "solution": solution,
            "image_key": str(image_key).replace("\\", "/"),
            "answer_type": rec.get("answer_type", ""),
            "image_organ": rec.get("image_organ", "") or rec.get("location", ""),
            "content_type": rec.get("content_type", "") or rec.get("question_type", ""),
        })

    if skipped:
        print(f"  [unify] {dataset}/{subset}/{split}: skipped {skipped} malformed record(s)")
    return out


# --------------------------------------------------------------------------- #
# 4. Image resolution
# --------------------------------------------------------------------------- #

_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def build_image_index(search_roots: Iterable[str]) -> Dict[str, str]:
    """
    Index every image under the given roots by several keys, so that whichever
    public image dataset is attached, lookups still resolve.

    Keys registered per file:
      * basename                     -> synpic54610.jpg
      * parent/basename              -> xmlab1/source.jpg   (SLAKE)
      * grandparent/parent/basename
    """
    index: Dict[str, str] = {}
    n = 0
    for root in search_roots:
        rp = Path(root)
        if not rp.exists():
            continue
        for p in rp.rglob("*"):
            if p.suffix.lower() not in _IMG_EXT or not p.is_file():
                continue
            n += 1
            parts = p.parts
            keys = [p.name.lower()]
            if len(parts) >= 2:
                keys.append(f"{parts[-2]}/{p.name}".lower())
            if len(parts) >= 3:
                keys.append(f"{parts[-3]}/{parts[-2]}/{p.name}".lower())
            for k in keys:
                index.setdefault(k, str(p))
    print(f"  [image index] indexed {n} image files -> {len(index)} keys")
    return index


def resolve_image(image_key: str, index: Dict[str, str]) -> Optional[str]:
    """Resolve a record's image reference against the index."""
    if not image_key:
        return None
    k = image_key.lower().lstrip("./")
    if k in index:
        return index[k]
    parts = k.split("/")
    for cand in ("/".join(parts[-2:]), parts[-1]):
        if cand in index:
            return index[cand]
    # SLAKE masks/sources sometimes differ in extension only.
    stem = Path(parts[-1]).stem
    for ext in _IMG_EXT:
        cand = f"{stem}{ext}"
        if len(parts) >= 2:
            two = f"{parts[-2]}/{cand}"
            if two in index:
                return index[two]
        if cand in index:
            return index[cand]
    return None


# --------------------------------------------------------------------------- #
# 5. Prompt construction (the three strategies of paper Fig. 1c)
# --------------------------------------------------------------------------- #

def format_options(options: Sequence[str]) -> str:
    if not options:
        return ""
    parts = [f"({OPTION_LETTERS[i]}) {o}" for i, o in enumerate(options)]
    return f"{MARK_OPT} " + " ".join(parts)


def build_input_text(rec: dict, strategy: str, generated_rationale: str = "") -> str:
    """
    Model input.

    Only 'Second-Stage_Reasoning' consumes a rationale on the input side; it
    receives the rationale produced by the stage-1 model, never the gold one
    (using gold at inference would leak the answer).
    """
    parts = [f"{MARK_Q} {rec['question']}"]
    opts = format_options(rec.get("options", []))
    if opts:
        parts.append(opts)
    if strategy == "Second-Stage_Reasoning":
        parts.append(f"{MARK_RAT} {bn_normalize(generated_rationale)}")
    return "\n".join(parts)


def build_target_text(rec: dict, strategy: str) -> str:
    """Training target for each strategy."""
    a, r = rec["answer"], rec.get("solution", "")
    if strategy == "without_R":
        return f"{MARK_ANS} {a}"
    if strategy == "Explanation":                 # A then R
        return f"{MARK_ANS} {a} {MARK_RAT} {r}"
    if strategy == "Reasoning":                   # R then A
        return f"{MARK_RAT} {r} {MARK_ANS} {a}"
    if strategy == "First-Stage_Reasoning":       # rationale only
        return r
    if strategy == "Second-Stage_Reasoning":      # answer only, rationale given
        return f"{MARK_ANS} {a}"
    raise ValueError(f"unknown strategy: {strategy}")


_ANS_RE = re.compile(re.escape(MARK_ANS) + r"\s*(.*?)(?:\s*" + re.escape(MARK_RAT) + r"|$)", re.S)
_RAT_RE = re.compile(re.escape(MARK_RAT) + r"\s*(.*?)(?:\s*" + re.escape(MARK_ANS) + r"|$)", re.S)


def parse_answer(text: str, strategy: str) -> Tuple[str, str]:
    """
    Split a generation into (answer, rationale).

    Works for both orderings because the markers are searched independently.
    If no marker survives generation, the whole string is treated as the
    answer for answer-bearing strategies.
    """
    if strategy == "First-Stage_Reasoning":
        return "", text.strip()
    a = _ANS_RE.search(text)
    r = _RAT_RE.search(text)
    answer = a.group(1).strip() if a else text.strip()
    rationale = r.group(1).strip() if r else ""
    return answer, rationale


# --------------------------------------------------------------------------- #
# 6. Torch dataset / collation
# --------------------------------------------------------------------------- #

class MedThinkDataset:
    """Pairs a unified record list with its cached DETR features."""

    def __init__(
        self,
        records: List[dict],
        tokenizer,
        cfg: Config,
        features: Optional[np.ndarray] = None,
        name_to_row: Optional[Dict[str, int]] = None,
        rationales: Optional[Dict[str, str]] = None,
    ):
        self.records = records
        self.tok = tokenizer
        self.cfg = cfg
        self.features = features
        self.name_to_row = name_to_row or {}
        self.rationales = rationales or {}
        self.missing = 0

    def __len__(self) -> int:
        return len(self.records)

    def image_feature(self, rec: dict) -> np.ndarray:
        """DETR features for this record, or zeros when the image is absent."""
        zero = np.zeros((self.cfg.patch_num, self.cfg.patch_dim), dtype=np.float32)
        if self.features is None:
            return zero
        row = self.name_to_row.get(rec["image_key"].lower())
        if row is None:
            self.missing += 1
            return zero
        feat = np.asarray(self.features[row], dtype=np.float32)
        # A single inf/nan from a corrupt image would poison the whole batch.
        return np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)

    def __getitem__(self, i: int) -> dict:
        rec = self.records[i]
        # Keyed by the globally unique id, not qid -- see record_uid().
        gen_rat = self.rationales.get(record_uid(rec), "")
        src = build_input_text(rec, self.cfg.strategy, gen_rat)
        tgt = build_target_text(rec, self.cfg.strategy)

        # No padding here: the collate function pads each batch to its own
        # longest member. Padding every example to source_len=512 wasted ~89%
        # of encoder positions -- real inputs are 11-78 tokens. source_len and
        # target_len now act purely as truncation caps.
        enc = self.tok(
            src, max_length=self.cfg.source_len,
            padding=False, truncation=True, return_tensors="np",
        )
        lab = self.tok(
            tgt, max_length=self.cfg.target_len,
            padding=False, truncation=True, return_tensors="np",
        )
        labels = lab["input_ids"][0].astype(np.int64)
        # -100 masks pad tokens out of the loss (Eq. 6 sums over real tokens).
        labels[labels == self.tok.pad_token_id] = -100
        # An entirely-masked row makes cross-entropy average over zero elements
        # and return NaN, which then contaminates the whole batch. It happens
        # for empty targets -- e.g. the one R-RAD open-end record with a blank
        # solution under First-Stage_Reasoning. Keep one real token.
        if not (labels != -100).any():
            labels[0] = self.tok.eos_token_id

        return {
            "input_ids": enc["input_ids"][0].astype(np.int64),
            "attention_mask": enc["attention_mask"][0].astype(np.int64),
            "labels": labels,
            "image_ids": self.image_feature(rec),
            "index": i,
        }


def collate_factory(pad_token_id: int = 0):
    """
    Pad each batch to its own longest member (dynamic padding).

    Attention masks and the -100 label mask make this exactly equivalent to
    fixed-length padding, at a fraction of the compute.
    """
    import torch

    def _pad(rows: List[np.ndarray], fill: int) -> "torch.Tensor":
        width = max(len(r) for r in rows)
        out = np.full((len(rows), width), fill, dtype=np.int64)
        for i, r in enumerate(rows):
            out[i, :len(r)] = r
        return torch.from_numpy(out)

    def collate(batch: List[dict]) -> dict:
        return {
            "input_ids": _pad([b["input_ids"] for b in batch], pad_token_id),
            "attention_mask": _pad([b["attention_mask"] for b in batch], 0),
            # -100 keeps padded target positions out of the loss (Eq. 6).
            "labels": _pad([b["labels"] for b in batch], -100),
            "image_ids": torch.from_numpy(np.stack([b["image_ids"] for b in batch])),
            "index": torch.tensor([b["index"] for b in batch]),
        }

    return collate


# --------------------------------------------------------------------------- #
# 7. Model: BanglaT5/mT5 + cross-attention + gated fusion (paper Sec. 3.2)
# --------------------------------------------------------------------------- #

def _torch_bits():
    import torch
    import torch.nn as nn
    from transformers import T5ForConditionalGeneration
    from transformers.modeling_outputs import BaseModelOutput
    return torch, nn, T5ForConditionalGeneration, BaseModelOutput


_torch, _nn, _T5Base, _BaseModelOutput = (None, None, None, None)
try:  # pragma: no cover - import-time convenience
    _torch, _nn, _T5Base, _BaseModelOutput = _torch_bits()
except Exception:  # torch absent (e.g. docs build)
    pass


if _T5Base is not None:

    class MedThinkMultimodalGeneration(_T5Base):  # type: ignore[misc,valid-type]
        """
        T5 conditional generation with MedThink's visual pathway bolted on.

        Text encoder output F_T is fused with DETR features F_I via
        cross-attention (Eq. 2) and a learned gate (Eq. 3-4) before the decoder
        ever sees it, so the decoder is unchanged and generation is unaffected.
        """

        def __init__(self, config, patch_dim: int = 256, patch_num: int = 100):
            super().__init__(config)
            d = config.d_model
            self.patch_dim = patch_dim
            self.patch_num = patch_num
            self.image_dense = _nn.Linear(patch_dim, d)
            self.mha_layer = _nn.MultiheadAttention(
                embed_dim=d, num_heads=1, kdim=d, vdim=d, batch_first=True
            )
            self.gate_dense = _nn.Linear(2 * d, d)
            self.sigmoid = _nn.Sigmoid()
            self._init_visual()

        def _init_visual(self):
            """
            Initialise the visual pathway so it starts *closed*.

            With a naive init the gate sits at sigmoid(0) = 0.5, so half the
            pretrained encoder output is replaced by a randomly-projected image
            signal at step 0. That wrecks the pretrained representation before
            the model has learned anything, and on a T5 v1.1 backbone
            (gated-gelu -- both mT5 and BanglaT5) it diverges to NaN within a
            few optimiser steps.

            Instead the gate bias starts at -3, so lambda ~= 0.047: the model
            begins as essentially pure text BanglaT5 and *learns* to open the
            gate to vision. The weight gets a small non-zero init rather than
            zeros -- with exactly zero weights every output channel would
            receive an identical gradient and the gate could never become
            channel-dependent.
            """
            # `from_pretrained` may build the model on the meta device
            # (low_cpu_mem_usage, the modern default). Initialising a meta
            # tensor is a silent no-op, so bail out here and let `load_model`
            # re-run this once the parameters are real.
            mods = [getattr(self, n, None)
                    for n in ("image_dense", "gate_dense", "mha_layer")]
            if any(m is None for m in mods):
                return                      # called before __init__ finished
            if any(p.is_meta for m in mods for p in m.parameters()):
                return                      # still on the meta device

            _nn.init.xavier_uniform_(self.image_dense.weight)
            _nn.init.zeros_(self.image_dense.bias)

            _nn.init.xavier_uniform_(self.gate_dense.weight)
            with _torch.no_grad():
                self.gate_dense.weight.mul_(0.01)   # keep lambda near the bias
            _nn.init.constant_(self.gate_dense.bias, -3.0)

            # nn.MultiheadAttention owns in_proj_weight / out_proj, which are
            # likewise absent from any T5 checkpoint.
            self.mha_layer._reset_parameters()
            # _reset_parameters() initialises in_proj_weight and the biases but
            # NOT out_proj.weight, which can therefore survive meta-device
            # loading as uninitialised memory -- finite by luck on one run and
            # enormous on the next. Initialise it explicitly.
            _nn.init.xavier_uniform_(self.mha_layer.out_proj.weight)
            if self.mha_layer.out_proj.bias is not None:
                _nn.init.zeros_(self.mha_layer.out_proj.bias)

        def _init_weights(self, module):
            """
            Extend T5's initialiser to cover the visual pathway.

            T5PreTrainedModel._init_weights only knows T5's own module types and
            silently ignores anything else, so without this the visual layers
            can survive `from_pretrained` as uninitialised memory -- which shows
            up as +-inf in image_dense and NaN everywhere downstream.
            """
            super()._init_weights(module)
            # This runs during super().__init__() too, before the visual layers
            # are assigned, so every lookup must tolerate their absence.
            visual = [getattr(self, n, None)
                      for n in ("image_dense", "gate_dense", "mha_layer")]
            if any(module is m for m in visual if m is not None):
                self._init_visual()

        def fuse(self, hidden_states, image_ids):
            """Eq. 2-4."""
            img = self.image_dense(image_ids.to(hidden_states.dtype))   # (B,100,d)
            att, _ = self.mha_layer(hidden_states, img, img)            # H_attn
            merge = _torch.cat([hidden_states, att], dim=-1)
            gate = self.sigmoid(self.gate_dense(merge))                 # lambda
            return (1 - gate) * hidden_states + gate * att              # F_fuse

        def encode_with_image(self, input_ids, attention_mask, image_ids):
            """Run the text encoder and fuse vision in. Returns encoder outputs."""
            enc = self.encoder(
                input_ids=input_ids, attention_mask=attention_mask, return_dict=True
            )
            fused = self.fuse(enc.last_hidden_state, image_ids)
            return _BaseModelOutput(last_hidden_state=fused)

        def forward(self, input_ids=None, attention_mask=None, image_ids=None,
                    encoder_outputs=None, labels=None, **kwargs):
            if encoder_outputs is None:
                encoder_outputs = self.encode_with_image(input_ids, attention_mask, image_ids)
            return super().forward(
                attention_mask=attention_mask,
                encoder_outputs=encoder_outputs,
                labels=labels,
                **kwargs,
            )

        @_torch.no_grad()
        def generate_with_image(self, input_ids, attention_mask, image_ids, **gen_kwargs):
            """
            Generate with vision fused in.

            The fused encoder output is computed once and handed to `generate`
            as `encoder_outputs`, so HF skips re-encoding. This avoids patching
            `prepare_inputs_for_generation`, which is the part of the original
            MM-CoT/MedThink code most likely to break across transformers
            versions.
            """
            enc = self.encode_with_image(input_ids, attention_mask, image_ids)
            return self.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                encoder_outputs=enc,
                **gen_kwargs,
            )

else:  # pragma: no cover
    class MedThinkMultimodalGeneration:  # type: ignore[no-redef]
        def __init__(self, *a, **k):
            raise ImportError("PyTorch/transformers required")


# Backwards-compatible alias (the class is backbone-agnostic).
BanglaT5MultimodalGeneration = MedThinkMultimodalGeneration


def load_model(cfg: Config, tokenizer=None):
    """Instantiate the backbone with the visual pathway and resize embeddings."""
    import torch

    model = MedThinkMultimodalGeneration.from_pretrained(
        cfg.model_name, patch_dim=cfg.patch_dim, patch_num=cfg.patch_num
    )
    if tokenizer is not None and len(tokenizer) != model.config.vocab_size:
        model.resize_token_embeddings(len(tokenizer))

    # The visual pathway is never present in a T5 checkpoint, so re-initialising
    # it here is always correct -- and necessary, because meta-device loading
    # can leave these tensors as uninitialised memory. Cheap insurance against
    # a silent +-inf that only surfaces as a NaN loss hours into training.
    model._init_visual()

    visual = {n: p for n, p in model.named_parameters()
              if n.startswith(("image_dense", "mha_layer", "gate_dense"))}
    bad = [n for n, p in visual.items() if not torch.isfinite(p).all()]
    if bad:
        raise RuntimeError(
            f"Non-finite visual parameters after initialisation: {bad}. "
            "This usually means meta-device loading bypassed _init_visual."
        )
    print(f"  visual pathway initialised: {len(visual)} tensors, all finite "
          f"(gate bias {model.gate_dense.bias.mean().item():.2f} "
          f"-> lambda ~ {torch.sigmoid(model.gate_dense.bias).mean().item():.3f})")
    return model


# --------------------------------------------------------------------------- #
# 8. Misc
# --------------------------------------------------------------------------- #

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def count_parameters(model) -> Dict[str, float]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    visual = sum(
        p.numel()
        for n, p in model.named_parameters()
        if n.startswith(("image_dense", "mha_layer", "gate_dense"))
    )
    return {
        "total_M": total / 1e6,
        "trainable_M": trainable / 1e6,
        "visual_head_M": visual / 1e6,
    }


# --------------------------------------------------------------------------- #
# 9. Training / inference engine
# --------------------------------------------------------------------------- #
# Shared by notebooks 02, 03 and 04 so the loop exists in exactly one place.
# Built for Kaggle: every run checkpoints each epoch and honours a wall-clock
# budget, so a session cut off at the 12h limit resumes instead of restarting.

def _build_optimizer(model, cfg: "Config", lr: float):
    import torch
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if (n.endswith("bias") or "layer_norm" in n.lower()) else decay).append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr,
    )


def _lr_lambda_factory(total_steps: int, warmup_ratio: float):
    warmup = max(1, int(total_steps * warmup_ratio))

    def fn(step: int) -> float:
        if step < warmup:
            return step / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return max(0.0, 1.0 - progress)

    return fn


def train_model(
    model,
    train_ds: "MedThinkDataset",
    cfg: "Config",
    lr: Optional[float] = None,
    epochs: Optional[int] = None,
    ckpt_path: Optional[str] = None,
    time_budget_h: float = 10.5,
    log_every: int = 50,
    resume: bool = True,
):
    """
    Fine-tune with the paper's objective (Eq. 6).

    Returns (model, info). `info` records per-epoch loss, elapsed time and
    whether the wall-clock budget cut the run short, so a partially trained
    result is never reported as if it were complete.
    """
    import time
    import torch
    from torch.utils.data import DataLoader

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    lr = cfg.lr if lr is None else lr
    epochs = cfg.resolved_epochs() if epochs is None else epochs

    loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        collate_fn=collate_factory(train_ds.tok.pad_token_id),
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(), drop_last=False,
    )
    steps_per_epoch = max(1, math.ceil(len(loader) / cfg.grad_accum))
    total_steps = steps_per_epoch * epochs

    optimizer = _build_optimizer(model, cfg, lr)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, _lr_lambda_factory(total_steps, cfg.warmup_ratio)
    )

    start_epoch, history = 0, []
    if resume and ckpt_path and os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location=device)
        saved = state.get("config", {})
        # A checkpoint from a quick_run was trained on a 200-example subset for
        # a couple of epochs. Silently resuming a full run from it would carry
        # those weights and epoch counter into the real result. Refuse.
        mismatch = [
            k for k in ("quick_run", "dataset", "subset", "strategy", "model_name")
            if k in saved and saved[k] != getattr(cfg, k)
        ]
        if mismatch:
            print(f"  [resume] IGNORING checkpoint -- differs in {mismatch}.")
            print(f"           Training from scratch. Delete {ckpt_path} to silence this.")
        else:
            model.load_state_dict(state["model"])
            optimizer.load_state_dict(state["optimizer"])
            scheduler.load_state_dict(state["scheduler"])
            start_epoch = state["epoch"] + 1
            history = state.get("history", [])
            print(f"  [resume] restored epoch {state['epoch']} -> continuing at {start_epoch}")

    t0 = time.time()
    stopped_early = False
    nan_batches = nan_steps = 0

    for epoch in range(start_epoch, epochs):
        model.train()
        running, nb = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(loader):
            batch = {k: v.to(device, non_blocking=True)
                     for k, v in batch.items() if k != "index"}
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                image_ids=batch["image_ids"],
                labels=batch["labels"],
            )

            # A non-finite loss must never be averaged into the epoch mean and
            # reported as if it were training. Drop the batch instead.
            if not torch.isfinite(out.loss):
                nan_batches += 1
                optimizer.zero_grad(set_to_none=True)
                if nan_batches == 1:
                    print("    [warn] non-finite loss -- batch skipped. "
                          "If this repeats, lower cfg.lr.")
                continue

            (out.loss / cfg.grad_accum).backward()
            running += out.loss.item()
            nb += 1

            if (step + 1) % cfg.grad_accum == 0 or (step + 1) == len(loader):
                gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                # Clipping does not rescue inf/nan gradients: stepping on them
                # turns every weight into NaN permanently.
                if torch.isfinite(gnorm):
                    optimizer.step()
                else:
                    nan_steps += 1
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if log_every and (step + 1) % (log_every * cfg.grad_accum) == 0:
                print(f"    ep {epoch+1}/{epochs} step {step+1}/{len(loader)} "
                      f"loss {running/max(nb,1):.4f} lr {scheduler.get_last_lr()[0]:.2e}")

        elapsed_h = (time.time() - t0) / 3600
        history.append({"epoch": epoch, "loss": running / max(nb, 1), "elapsed_h": elapsed_h})
        print(f"  epoch {epoch+1}/{epochs}  loss {history[-1]['loss']:.4f}  ({elapsed_h:.2f}h)")

        if ckpt_path:
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "history": history,
                "config": asdict(cfg),
            }, ckpt_path)

        if elapsed_h > time_budget_h:
            print(f"  [budget] {elapsed_h:.2f}h exceeds {time_budget_h}h budget -- stopping "
                  f"at epoch {epoch+1}/{epochs}. Re-run the notebook to resume.")
            stopped_early = True
            break

    if nan_batches or nan_steps:
        print(f"\n  [warn] {nan_batches} non-finite batches, {nan_steps} skipped steps. "
              "Treat these results as unreliable and lower cfg.lr.")

    return model, {
        "history": history,
        "epochs_completed": (history[-1]["epoch"] + 1) if history else 0,
        "epochs_target": epochs,
        "stopped_early": stopped_early,
        "nan_batches": nan_batches,
        "nan_steps": nan_steps,
        "diverged": bool(history) and not math.isfinite(history[-1]["loss"]),
    }


def diagnose_forward(model, ds: "MedThinkDataset", cfg: "Config", n: int = 4) -> bool:
    """
    Run one forward pass and report where non-finite values first appear.

    Pinpoints a NaN instead of guessing: if the encoder output is already bad
    the problem is the backbone or the input; if only `fused` is bad it is the
    visual pathway; if only the loss is bad it is the labels.
    Returns True when everything is finite.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    batch = collate_factory(ds.tok.pad_token_id)([ds[i] for i in range(min(n, len(ds)))])
    batch = {k: v.to(device) for k, v in batch.items() if k != "index"}

    ok = True
    with torch.no_grad():
        stages = {}
        stages["image_ids"] = batch["image_ids"]
        enc = model.encoder(input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"], return_dict=True)
        stages["encoder_out"] = enc.last_hidden_state
        stages["image_dense"] = model.image_dense(batch["image_ids"])
        att, _ = model.mha_layer(enc.last_hidden_state,
                                 stages["image_dense"], stages["image_dense"])
        stages["cross_attn"] = att
        merge = torch.cat([enc.last_hidden_state, att], dim=-1)
        stages["gate"] = model.sigmoid(model.gate_dense(merge))
        stages["fused"] = model.fuse(enc.last_hidden_state, batch["image_ids"])

        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    image_ids=batch["image_ids"], labels=batch["labels"])
        stages["logits"] = out.logits
        stages["loss"] = out.loss

        print(f"{'stage':14s}{'finite':>8s}{'mean':>12s}{'std':>12s}{'absmax':>12s}")
        print("-" * 58)
        for name, t in stages.items():
            fin = bool(torch.isfinite(t).all())
            ok &= fin
            f = t[torch.isfinite(t)].float()
            print(f"{name:14s}{('yes' if fin else 'NO'):>8s}"
                  f"{(f.mean().item() if f.numel() else float('nan')):>12.4f}"
                  f"{(f.std().item() if f.numel() > 1 else 0.0):>12.4f}"
                  f"{(f.abs().max().item() if f.numel() else float('nan')):>12.2f}")

    lam = stages["gate"].float().mean().item()
    print(f"\nmean gate lambda = {lam:.4f}  (should start near 0.05: the visual")
    print("pathway opens gradually rather than overwriting the pretrained encoder)")
    print("\nALL FINITE" if ok else "\nNON-FINITE VALUES PRESENT -- see the first 'NO' above")
    return ok


def generate_predictions(
    model,
    ds: "MedThinkDataset",
    cfg: "Config",
    batch_size: Optional[int] = None,
    max_new_tokens: Optional[int] = None,
) -> List[str]:
    """Decode over a dataset, returned in original record order."""
    import torch
    from torch.utils.data import DataLoader

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    loader = DataLoader(
        ds, batch_size=batch_size or max(cfg.batch_size * 2, 8), shuffle=False,
        collate_fn=collate_factory(ds.tok.pad_token_id),
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    preds: List[Tuple[int, str]] = []

    with torch.no_grad():
        for batch in loader:
            idx = batch["index"].tolist()
            out = model.generate_with_image(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                image_ids=batch["image_ids"].to(device),
                max_length=max_new_tokens or cfg.target_len,
                num_beams=cfg.num_beams,
                early_stopping=cfg.num_beams > 1,
            )
            preds.extend(zip(idx, ds.tok.batch_decode(out, skip_special_tokens=True)))

    preds.sort(key=lambda x: x[0])
    return [t for _, t in preds]


def measure_gate(model, ds: "MedThinkDataset", cfg: "Config", n: int = 64) -> Dict[str, float]:
    """
    How far has the visual gate opened?

    lambda starts at ~0.047 by construction. If it is still ~0.047 after
    training, the model has learned to ignore the images and is a text-only
    model wearing a multimodal architecture -- which would invalidate any
    claim about rationale-guided *multimodal* reasoning. Report this.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    batch = collate_factory(ds.tok.pad_token_id)([ds[i] for i in range(min(n, len(ds)))])

    with torch.no_grad():
        enc = model.encoder(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            return_dict=True,
        )
        img = model.image_dense(batch["image_ids"].to(device).to(enc.last_hidden_state.dtype))
        att, _ = model.mha_layer(enc.last_hidden_state, img, img)
        gate = model.sigmoid(model.gate_dense(
            torch.cat([enc.last_hidden_state, att], dim=-1)))
        # Mask out padding positions, which carry no signal.
        mask = batch["attention_mask"].to(device).unsqueeze(-1).bool()
        real = gate.masked_select(mask)

    return {
        "gate/lambda_mean": real.mean().item(),
        "gate/lambda_max": real.max().item(),
        "gate/shift_from_init": real.mean().item() - 0.047,
    }


def majority_baseline(records: List[dict]) -> float:
    """
    Accuracy of always predicting the most common answer.

    The number every closed-end result must beat. A yes/no test set is rarely
    balanced, so raw accuracy alone cannot distinguish a model that learned
    the task from one that learned the class prior.
    """
    if not records:
        return 0.0
    counts = Counter(r["answer"] for r in records)
    return 100.0 * max(counts.values()) / len(records)


def seed_checkpoint(ckpt_path: str, data_root: str = "/kaggle/input") -> bool:
    """
    Copy a previous session's checkpoint into the writable working directory.

    Kaggle wipes /kaggle/working between sessions, so `train_model`'s resume
    finds nothing and a multi-session run silently restarts from epoch 0 --
    burning the hours it was supposed to save. If the previous session's output
    is attached as a dataset, its checkpoint is found here and copied in.

    Returns True when a checkpoint was seeded.
    """
    import shutil

    if os.path.exists(ckpt_path):
        return False                      # this session already has one
    name = os.path.basename(ckpt_path)
    hits = sorted(Path(data_root).glob(f"**/{name}"))
    if not hits:
        return False
    os.makedirs(os.path.dirname(ckpt_path) or ".", exist_ok=True)
    shutil.copy(str(hits[0]), ckpt_path)
    print(f"  [seed] restored checkpoint from {hits[0]}")
    return True


def record_uid(rec: dict) -> str:
    """
    Globally unique record id: dataset/subset/split/qid.

    `qid` is only unique *within one JSON file*. R-PathVQA numbers its train and
    test splits from 1 independently, so 3257 closed-end qids (and 3292
    open-end) appear in both. Keying the two-stage rationale cache by qid alone
    silently overwrites ~35% of the training rationales with test-derived ones
    -- both a correctness bug and train/test contamination. R-RAD and R-SLAKE
    happen not to collide, which is precisely why this hides until PathVQA.
    """
    return "/".join(str(rec.get(k, "")) for k in ("dataset", "subset", "split", "qid"))


def degenerate_rate(answer_preds: Sequence[str]) -> Dict[str, float]:
    """
    How often the model produced no actual answer.

    A model that emits only the "উত্তর:" marker and stops still scores ~50
    ROUGE-1 against a short `without_R` target, because the marker is half the
    reference. That looks like a result and is not one. This measures the
    failure directly: what fraction of answer spans are empty or punctuation.
    """
    n = max(len(answer_preds), 1)
    empty = sum(1 for a in answer_preds if not strip_punct(bn_normalize(a)))
    lens = [len(bn_tokenize(a)) for a in answer_preds]
    return {
        "EmptyAnswerRate": 100.0 * empty / n,
        "MeanAnswerTokens": sum(lens) / n,
    }


def constant_answer_baseline(train_recs: List[dict],
                             test_recs: List[dict]) -> Tuple[str, Dict[str, float]]:
    """
    Score of always emitting the single most common training answer.

    The open-end analogue of the majority-class baseline. Open-end BLEU/ROUGE
    can look respectable purely from reproducing the answer *style* without
    answering anything, so this is the floor a real result must clear.

    Returns (the constant answer, its metrics).
    """
    if not train_recs or not test_recs:
        return "", {}
    const = Counter(r["answer"] for r in train_recs).most_common(1)[0][0]
    scores = evaluate_open_end([const] * len(test_recs),
                               [r["answer"] for r in test_recs])
    return const, {f"baseline/{k}": v for k, v in scores.items()}


def save_results(cfg: "Config", metrics: Dict[str, float], records: List[dict],
                 predictions: List[str], train_info: dict,
                 out_dir: Optional[str] = None) -> str:
    """Write metrics + per-example predictions for notebook 05 to aggregate."""
    out_dir = out_dir or cfg.output_dir
    os.makedirs(out_dir, exist_ok=True)
    name = cfg.run_name()

    mpath = os.path.join(out_dir, f"metrics_{name}.json")
    with open(mpath, "w", encoding="utf-8") as fh:
        json.dump({"run": name, "config": asdict(cfg), "metrics": metrics,
                   "train": train_info}, fh, ensure_ascii=False, indent=2)

    ppath = os.path.join(out_dir, f"preds_{name}.jsonl")
    with open(ppath, "w", encoding="utf-8") as fh:
        for rec, pred in zip(records, predictions):
            ans, rat = parse_answer(pred, cfg.strategy)
            fh.write(json.dumps({
                "qid": rec["qid"],
                "question": rec["question"],
                "gold_answer": rec["answer"],
                "options": rec["options"],
                "pred_raw": pred,
                "pred_answer": ans,
                "pred_rationale": rat,
                "gold_rationale": rec["solution"],
                "image_key": rec["image_key"],
                "image_organ": rec.get("image_organ", ""),
            }, ensure_ascii=False) + "\n")

    print(f"  wrote {mpath}")
    print(f"  wrote {ppath}")
    return mpath

Writing medthink_bn.py


In [2]:
import os, sys, json, glob, math, time
from pathlib import Path
import numpy as np

# Optional: BanglaT5's official normaliser. Falls back gracefully if offline.
try:
    import normalizer  # noqa: F401
except ImportError:
    !pip install -q git+https://github.com/csebuetnlp/normalizer.git 2>/dev/null || echo "normalizer unavailable - using NFKC fallback"

import importlib, medthink_bn
importlib.reload(medthink_bn)
import medthink_bn as M

print("csebuetnlp normalizer:", M._HAS_CSEBUETNLP)
DATA_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import torch
from transformers import AutoTokenizer
print("cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

def load_manifest(ds, sub, split):
    c = sorted(Path(DATA_ROOT).glob(f"**/manifests/{ds}_{sub}_{split}.json"))
    assert c, f"manifest {ds}_{sub}_{split}.json not found - attach medthink-bn-manifests"
    return json.load(open(c[0], encoding="utf-8"))

def load_features():
    f = sorted(Path(DATA_ROOT).glob("**/features.npy"))
    n = sorted(Path(DATA_ROOT).glob("**/name_to_row.json"))
    assert f and n, "features not found - attach medthink-bn-features"
    return np.load(f[0], mmap_mode="r"), json.load(open(n[0], encoding="utf-8"))

def subsample(recs, n, seed=0):
    import random as _r
    if n is None or n >= len(recs):
        return recs
    g = _r.Random(seed)
    return g.sample(recs, n)

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
csebuetnlp normalizer: True
cuda: True | Tesla T4


## ⚙️ Config

In [3]:
cfg1 = M.Config(
    dataset    = "slake",
    subset     = "closed-end",          # closed-end | open-end
    strategy   = "First-Stage_Reasoning",
    epochs     = None,                  # paper value for the dataset
    lr         = 1e-4,                  # not the paper's 5e-4: T5 v1.1 diverges
    batch_size = 8, grad_accum = 4,
    target_len = 320, seed = 42,
    quick_run  = False,                  # <<<< SET False FOR THE REAL RUN
)

TIME_BUDGET_H = 5.0                     # per stage; two stages in one session
if cfg1.quick_run:
    cfg1.epochs = 2

import dataclasses
cfg2 = dataclasses.replace(
    cfg1,
    strategy="Second-Stage_Reasoning",
    lr=5e-5,                            # paper Sec. 5.1
    epochs=2 if cfg1.quick_run else 20, # paper Sec. 5.1
    target_len=64,                      # stage 2 emits an answer only
)
print("stage 1:", cfg1.run_name(), "| stage 2:", cfg2.run_name())

stage 1: slake_closed-end_First-Stage_Reasoning_banglat5 | stage 2: slake_closed-end_Second-Stage_Reasoning_banglat5


## 1 · Data

In [4]:
tokenizer = AutoTokenizer.from_pretrained(cfg1.model_name, use_fast=False)
features, name_to_row = load_features()

train_recs = load_manifest(cfg1.dataset, cfg1.subset, "train")
test_recs  = load_manifest(cfg1.dataset, cfg1.subset, "test")
if cfg1.quick_run:
    train_recs = subsample(train_recs, 200, cfg1.seed)
    test_recs  = subsample(test_recs, 100, cfg1.seed)
print(f"train {len(train_recs)} | test {len(test_recs)}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

train 1943 | test 416


## 2 · Stage 1 — (Q, I) → R

In [5]:
M.set_seed(cfg1.seed)
s1_train = M.MedThinkDataset(train_recs, tokenizer, cfg1, features, name_to_row)
S1_CKPT = f"{WORK}/ckpt_{cfg1.run_name()}.pt"
M.seed_checkpoint(S1_CKPT, DATA_ROOT)   # multi-session resume
model1 = M.load_model(cfg1, tokenizer)

assert M.diagnose_forward(model1, s1_train, cfg1),     "Non-finite forward pass — do not train. Check the first 'NO' row above."

model1, info1 = M.train_model(
    model1, s1_train, cfg1,
    ckpt_path=S1_CKPT,
    time_budget_h=TIME_BUDGET_H,
)
print("\nstage 1:", {k: v for k, v in info1.items() if k != "history"})
if info1.get("diverged") or info1.get("nan_batches"):
    print("!! STAGE 1 DIVERGED -- the rationales below are noise; stop here.")
if info1["stopped_early"]:
    raise RuntimeError(
        "Stage 1 hit TIME_BUDGET_H and is INCOMPLETE. Its checkpoint is saved -- "
        "re-run this notebook to resume stage 1. Continuing now would train "
        "stage 2 on rationales from a half-trained model."
    )

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MedThinkMultimodalGeneration LOAD REPORT from: csebuetnlp/banglat5
Key                       | Status  | 
--------------------------+---------+-
image_dense.bias          | MISSING | 
mha_layer.out_proj.bias   | MIS

  visual pathway initialised: 8 tensors, all finite (gate bias -3.00 -> lambda ~ 0.047)


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

stage           finite        mean         std      absmax
----------------------------------------------------------
image_ids          yes      0.0372      0.9639        7.29
encoder_out        yes     -0.0078      0.1150        1.24
image_dense        yes      0.0080      0.6996        2.55
cross_attn         yes     -0.0071      0.4420        1.40
gate               yes      0.0474      0.0002        0.05
fused              yes     -0.0078      0.1111        1.15
logits             yes      0.2110     21.9513      128.89
loss               yes    101.8698      0.0000      101.87

mean gate lambda = 0.0474  (should start near 0.05: the visual
pathway opens gradually rather than overwriting the pretrained encoder)

ALL FINITE
    ep 1/300 step 200/243 loss 89.3950 lr 4.55e-06
  epoch 1/300  loss 87.5562  (0.03h)
    ep 2/300 step 200/243 loss 60.0384 lr 1.01e-05
  epoch 2/300  loss 56.9546  (0.05h)
    ep 3/300 step 200/243 loss 29.0314 lr 1.57e-05
  epoch 3/300  loss 27.4236  (0.08h

RuntimeError: Stage 1 hit TIME_BUDGET_H and is INCOMPLETE. Its checkpoint is saved -- re-run this notebook to resume stage 1. Continuing now would train stage 2 on rationales from a half-trained model.

### 2b · Generate rationales for train and test

In [ ]:
s1_test = M.MedThinkDataset(test_recs, tokenizer, cfg1, features, name_to_row)

# Generating rationales costs up to ~5h on R-PathVQA and is pure inference,
# so cache it: a session that dies in stage 2 must not have to redo it.
RAT_PATH = f"{WORK}/rationales_{cfg1.run_name()}.json"
_cached = ([RAT_PATH] if os.path.exists(RAT_PATH) else
           [str(x) for x in sorted(Path(DATA_ROOT).glob(f"**/rationales_{cfg1.run_name()}.json"))])

if _cached:
    rationales = json.load(open(_cached[0], encoding="utf-8"))
    print(f"reusing cached rationales from {_cached[0]} ({len(rationales)} entries)")
    rat_test = [rationales[M.record_uid(r)] for r in test_recs]
else:
    rat_train = M.generate_predictions(model1, s1_train, cfg1)
    rat_test  = M.generate_predictions(model1, s1_test,  cfg1)

    # Keyed by the globally unique id, NOT qid: R-PathVQA numbers train and test
    # from 1 independently, so 3257 qids collide and a qid-keyed dict would
    # overwrite ~35% of the training rationales with test-derived ones.
    rationales = {M.record_uid(r): t for r, t in zip(train_recs, rat_train)}
    rationales.update({M.record_uid(r): t for r, t in zip(test_recs, rat_test)})
    assert len(rationales) == len(train_recs) + len(test_recs), (
        "rationale key collision -- train rationales were overwritten by test ones"
    )
    with open(RAT_PATH, "w", encoding="utf-8") as fh:
        json.dump(rationales, fh, ensure_ascii=False)
    print(f"cached {len(rationales)} rationales -> {RAT_PATH}")

# Quality of the stage-1 rationales themselves.
r_metrics = M.evaluate_open_end(rat_test, [r["solution"] for r in test_recs])
print("stage-1 rationale quality vs gold MDMR:")
for k, v in r_metrics.items():
    print(f"  {k:12s} {v:6.2f}")

print("\nexample:", rat_test[0][:250])
del model1
torch.cuda.empty_cache()

## 3 · Stage 2 — (Q, I, R) → A

In [ ]:
M.set_seed(cfg2.seed)
s2_train = M.MedThinkDataset(train_recs, tokenizer, cfg2, features, name_to_row,
                             rationales=rationales)
s2_test  = M.MedThinkDataset(test_recs,  tokenizer, cfg2, features, name_to_row,
                             rationales=rationales)

print("--- stage-2 input ---\n" +
      M.build_input_text(train_recs[0], cfg2.strategy, rationales[M.record_uid(train_recs[0])])[:400])

S2_CKPT = f"{WORK}/ckpt_{cfg2.run_name()}.pt"
M.seed_checkpoint(S2_CKPT, DATA_ROOT)
model2 = M.load_model(cfg2, tokenizer)
model2, info2 = M.train_model(
    model2, s2_train, cfg2,
    lr=cfg2.lr, epochs=cfg2.epochs,
    ckpt_path=S2_CKPT,
    time_budget_h=TIME_BUDGET_H,
)
print("\nstage 2:", {k: v for k, v in info2.items() if k != "history"})
if info2.get("diverged") or info2.get("nan_batches"):
    print("!! STAGE 2 DIVERGED. Results below are meaningless.")

## 4 · Evaluate

In [ ]:
preds = M.generate_predictions(model2, s2_test, cfg2)
pred_answers = [M.parse_answer(p, cfg2.strategy)[0] for p in preds]
gold = [r["answer"] for r in test_recs]

if cfg2.subset == "closed-end":
    metrics = M.evaluate_closed_end(pred_answers, gold, [r["options"] for r in test_recs])
    metrics["MajorityBaseline"] = M.majority_baseline(test_recs)
    metrics["LiftOverMajority"] = metrics["Accuracy"] - metrics["MajorityBaseline"]
    print("\npaper (Two-Stage, closed-end): R-RAD 80.5 | R-SLAKE 79.1 | R-Path 87.2")
else:
    full_refs = [M.build_target_text(r, cfg2.strategy) for r in test_recs]
    metrics = M.evaluate_open_end_dual(preds, full_refs, pred_answers, gold)
    metrics.update(M.degenerate_rate(pred_answers))
    const_ans, base = M.constant_answer_baseline(train_recs, test_recs)
    metrics.update(base)

# Did the visual gate open? Reported for stage 2, the model that answers.
metrics.update(M.measure_gate(model2, s2_test, cfg2))

for k, v in metrics.items():
    print(f"  {k:24s} {v:6.2f}")

for k, v in r_metrics.items():
    metrics[f"stage1_rationale/{k}"] = v

## 5 · Save

In [ ]:
info = {"stage1": info1, "stage2": info2}
M.save_results(cfg2, metrics, test_recs, preds, info)

with open(f"{WORK}/rationales_{cfg1.run_name()}.json", "w", encoding="utf-8") as fh:
    json.dump(rationales, fh, ensure_ascii=False)
print("saved generated rationales for reuse/inspection")